In [24]:
import os
from dotenv import load_dotenv

In [25]:
from openai import OpenAI
client = OpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"]
)

In [26]:
from ingest import load_documents,build_index
from rag_helper import RAGHelper

documents = load_documents()
index = build_index(documents)

In [28]:
instruction = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

assistant = RAGHelper(
    index = index,
    client = client,
    instructions= instruction
)

In [29]:
messages = [
    {
        "role": "user",
        "content": "How does the agentic loop work, and how is it different from plain RAG?"
    }
]
response = client.responses.create(
    model = os.environ["model"],
    input = messages
)

response.output_text

''

In [31]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [40]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


In [41]:
def search(query: str) -> list[dict[str, str]]:
    """
    Search the FAQ database for entries matching the given query.
    Use this to find relevant course material before answering a question.
    """
    return index.search(
        query,
        num_results=5
    )

agent_tools = Tools()
agent_tools.add_tool(search)


In [42]:

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.\nUse this to find relevant course material before answering a question.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [43]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [44]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instruction,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model=os.environ["model"],
        client=client
    )
)


In [45]:
result  = runner.loop(
    prompt = "How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback
)

-> Response received


/Users/fahmimshahriar/Documents/Project/llm-datatalkclub/.venv/lib/python3.13/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
